# Amazon Review Sentiment: Traditional ML and BERT

**Author:** Blessing Osuagwu  
**Project type:** NLP model comparison

This project compares TF-IDF baselines with a fine-tuned BERT classifier for binary sentiment classification. Ratings 1–2 are labeled negative, ratings 4–5 positive, and neutral rating-3 reviews are excluded. This label rule is a project choice, not a ground-truth sentiment annotation.

## Dataset

Download the CSV from the cited [Kaggle Amazon Review Sentiment Analysis dataset](https://www.kaggle.com/datasets/henryshan/amazon-review-sentiment-analysis) and place `amazon_reviews.csv` in `data/` or `/content/`. The dataset is not included here. Review its license and terms before redistribution.

## 1. Setup and load data

Install dependencies from `requirements.txt`. BERT fine-tuning works best with a GPU runtime; the TF-IDF baselines are lighter.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import classification_report, confusion_matrix, f1_score, accuracy_score

SEED = 42
DATA_FILE = Path("data/amazon_reviews.csv")
if not DATA_FILE.exists():
    DATA_FILE = Path("/content/amazon_reviews.csv")
if not DATA_FILE.exists():
    raise FileNotFoundError("Download amazon_reviews.csv from the Kaggle source linked in the README.")

raw = pd.read_csv(DATA_FILE)
required = {"reviewText", "overall"}
if required - set(raw.columns):
    raise ValueError(f"Expected columns {sorted(required)}; found {list(raw.columns)}")
print("Rows and columns:", raw.shape)
raw[["reviewText", "overall"]].head()

In [ ]:
df = raw[["reviewText", "overall"]].copy()
df["reviewText"] = df["reviewText"].fillna("").astype(str).str.strip()
df["overall"] = pd.to_numeric(df["overall"], errors="coerce")
df = df.dropna(subset=["overall"])
df = df[df["reviewText"].ne("")]
df = df[df["overall"].isin([1, 2, 3, 4, 5])].copy()
df["label"] = np.select(
    [df["overall"].le(2), df["overall"].ge(4)],
    [0, 1],
    default=-1,
)
df = df[df["label"].ne(-1)].reset_index(drop=True)
df["sentiment"] = df["label"].map({0: "negative", 1: "positive"})
print("Usable binary-label reviews:", len(df))
display(df["sentiment"].value_counts())
print("Class proportions:\n", df["sentiment"].value_counts(normalize=True).round(3))

## 2. Exploratory analysis

The class balance is important: a model can achieve high accuracy by mostly predicting the majority class. Compare class-level metrics and macro F1, not accuracy alone.

In [ ]:
ax = df["sentiment"].value_counts().reindex(["negative", "positive"]).plot(
    kind="bar", color=["#e45756", "#54a24b"], title="Review labels after neutral ratings are excluded"
)
ax.set_xlabel("Sentiment"); ax.set_ylabel("Reviews"); plt.tight_layout(); plt.show()

df["review_length"] = df["reviewText"].str.split().str.len()
sns.boxplot(data=df, x="sentiment", y="review_length", showfliers=False)
plt.title("Review length by label"); plt.xlabel("Sentiment"); plt.ylabel("Words per review")
plt.tight_layout(); plt.show()

## 3. One shared train/validation/test split

All models use the same stratified split. The test set is held back until final evaluation; it is not used for early stopping, threshold selection, or model selection.

In [ ]:
train_val, test_df = train_test_split(
    df, test_size=0.20, stratify=df["label"], random_state=SEED
)
train_df, val_df = train_test_split(
    train_val, test_size=0.125, stratify=train_val["label"], random_state=SEED
)
print({"train": len(train_df), "validation": len(val_df), "test": len(test_df)})
for name, part in [("train", train_df), ("validation", val_df), ("test", test_df)]:
    print(name, part["label"].value_counts(normalize=True).round(3).to_dict())

## 4. Traditional ML baselines

The vectorizer is placed inside each pipeline so its vocabulary and IDF values are learned from the training split only. Class weighting is used for the linear models rather than synthesizing interpolated text vectors with SMOTE.

In [ ]:
models = {
    "Logistic Regression": Pipeline([
        ("tfidf", TfidfVectorizer(ngram_range=(1, 2), min_df=2, max_features=50000, sublinear_tf=True)),
        ("model", LogisticRegression(max_iter=1000, class_weight="balanced", random_state=SEED)),
    ]),
    "Linear SVM": Pipeline([
        ("tfidf", TfidfVectorizer(ngram_range=(1, 2), min_df=2, max_features=50000, sublinear_tf=True)),
        ("model", LinearSVC(class_weight="balanced", random_state=SEED)),
    ]),
    "Multinomial Naive Bayes": Pipeline([
        ("tfidf", TfidfVectorizer(ngram_range=(1, 2), min_df=2, max_features=50000, sublinear_tf=True)),
        ("model", MultinomialNB(fit_prior=False)),
    ]),
}

baseline_rows = []
for name, pipeline in models.items():
    pipeline.fit(train_df["reviewText"], train_df["label"])
    val_pred = pipeline.predict(val_df["reviewText"])
    baseline_rows.append({
        "model": name,
        "validation_accuracy": accuracy_score(val_df["label"], val_pred),
        "validation_macro_f1": f1_score(val_df["label"], val_pred, average="macro"),
    })
pd.DataFrame(baseline_rows).sort_values("validation_macro_f1", ascending=False)

## 5. Fine-tune BERT (optional, GPU recommended)

This section uses the same train and validation rows as the traditional models. It does not use the held-out test set during training. BERT fine-tuning can take substantially longer and use more memory than the TF-IDF baselines.

In [ ]:
RUN_BERT = False  # Set True to fine-tune BERT (GPU recommended; may take a while).
if RUN_BERT:
    import torch
    from torch.utils.data import Dataset
    from transformers import (
        AutoTokenizer, AutoModelForSequenceClassification,
        DataCollatorWithPadding, Trainer, TrainingArguments,
    )

    MODEL_NAME = "bert-base-uncased"
    MAX_LENGTH = 128
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

    class ReviewDataset(Dataset):
        def __init__(self, frame):
            self.texts = frame["reviewText"].tolist()
            self.labels = frame["label"].astype(int).tolist()
        def __len__(self):
            return len(self.labels)
        def __getitem__(self, index):
            item = tokenizer(self.texts[index], truncation=True, max_length=MAX_LENGTH)
            item["labels"] = self.labels[index]
            return item

    bert_train = ReviewDataset(train_df)
    bert_val = ReviewDataset(val_df)
    bert_test = ReviewDataset(test_df)
    bert_model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME, num_labels=2,
        id2label={0: "negative", 1: "positive"},
        label2id={"negative": 0, "positive": 1},
    )


In [ ]:
if RUN_BERT:
    def compute_metrics(eval_pred):
        logits, labels = eval_pred
        predictions = np.argmax(logits, axis=1)
        return {"macro_f1": f1_score(labels, predictions, average="macro")}

    training_args = TrainingArguments(
        output_dir="bert-review-checkpoints",
        num_train_epochs=3,
        learning_rate=2e-5,
        per_device_train_batch_size=8,
        per_device_eval_batch_size=16,
        gradient_accumulation_steps=2,
        eval_strategy="epoch",
        save_strategy="epoch",
        load_best_model_at_end=True,
        metric_for_best_model="eval_macro_f1",
        greater_is_better=True,
        weight_decay=0.01,
        report_to="none",
        seed=SEED,
    )
    trainer = Trainer(
        model=bert_model,
        args=training_args,
        train_dataset=bert_train,
        eval_dataset=bert_val,
        processing_class=tokenizer,
        data_collator=DataCollatorWithPadding(tokenizer=tokenizer),
        compute_metrics=compute_metrics,
    )
    trainer.train()
else:
    print("BERT skipped. Set RUN_BERT=True to fine-tune it; TF-IDF baselines remain available.")


## 6. Final test comparison

After choosing hyperparameters from validation results, evaluate each final model on the test split once. Treat the negative-class recall and macro F1 as important because negative reviews are the minority class in the original run.

In [ ]:
test_rows = []
for name, pipeline in models.items():
    pred = pipeline.predict(test_df["reviewText"])
    report = classification_report(test_df["label"], pred, output_dict=True, zero_division=0)
    test_rows.append({
        "model": name,
        "accuracy": report["accuracy"],
        "negative_recall": report["0"]["recall"],
        "macro_f1": report["macro avg"]["f1-score"],
    })

if RUN_BERT:
    bert_output = trainer.predict(bert_test)
    bert_pred = np.argmax(bert_output.predictions, axis=1)
    bert_report = classification_report(test_df["label"], bert_pred, output_dict=True, zero_division=0)
    test_rows.append({
        "model": "BERT",
        "accuracy": bert_report["accuracy"],
        "negative_recall": bert_report["0"]["recall"],
        "macro_f1": bert_report["macro avg"]["f1-score"],
    })

results = pd.DataFrame(test_rows).sort_values("macro_f1", ascending=False)
display(results)


## 7. Limitations and interpretation

The labels are derived from star ratings, and neutral reviews are excluded. Ratings are an imperfect proxy for written sentiment. The dataset is imbalanced, and aggregate results may not generalize to other product categories, time periods, or review sources. Compare macro F1 and minority-class recall alongside accuracy. Do not publish the metrics from the old HTML export as results for this cleaned workflow; this notebook uses a shared split and validation-only model selection.